# Transformaciones (Transforms)

Los datos no siempre vienen en su forma procesada final requerida para entrenar algoritmos de aprendizaje automático. Utilizamos **transformaciones** para realizar manipulaciones de los datos y hacerlos adecuados para el entrenamiento.

Todos los datasets de TorchVision tienen dos parámetros:
- **`transform`**: para modificar las características (features)
- **`target_transform`**: para modificar las etiquetas (labels)

Estos parámetros aceptan callables que contienen la lógica de transformación. El módulo `torchvision.transforms` ofrece varias transformaciones comúnmente utilizadas.

## Ejemplo con FashionMNIST

Las características de FashionMNIST están en formato de imagen PIL, y las etiquetas son enteros. Para el entrenamiento, necesitamos:
- Las características como **tensores normalizados**
- Las etiquetas como **tensores codificados en one-hot**

Para hacer estas transformaciones, utilizamos `ToTensor` y `Lambda`.

## Importar Librerías

In [ ]:
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor, Lambda

## Cargar Dataset con Transformaciones

In [ ]:
ds = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
    target_transform=Lambda(lambda y: torch.zeros(10, dtype=torch.float).scatter_(0, torch.tensor(y), value=1))
)

## ToTensor()

`ToTensor` convierte una imagen PIL o un array NumPy (`ndarray`) en un `FloatTensor`, y escala los valores de intensidad de píxeles de la imagen al rango **[0., 1.]**.

### ¿Qué hace ToTensor?

1. **Conversión de tipo**: Transforma imágenes PIL o arrays NumPy en tensores de PyTorch
2. **Cambio de formato**: Reordena las dimensiones de `(H x W x C)` a `(C x H x W)`
   - H = Alto (Height)
   - W = Ancho (Width)  
   - C = Canales (Channels)
3. **Normalización automática**: Divide los valores de píxeles por 255, llevándolos del rango [0, 255] al rango [0.0, 1.0]
4. **Conversión de dtype**: Cambia de `uint8` a `float32`

Esta normalización es importante porque:
- Los modelos de redes neuronales trabajan mejor con valores pequeños
- Facilita el entrenamiento y la convergencia
- Estandariza la entrada independientemente del formato original

## ¿Qué es One-Hot Encoding?

**One-Hot Encoding** (codificación uno-caliente) es una técnica para representar datos categóricos como vectores binarios. En vez de usar números para representar categorías, se crea un vector donde:
- **Solo una posición tiene el valor 1** (está "caliente" o "hot")
- **Todas las demás posiciones tienen el valor 0** (están "frías" o "cold")

### ¿Por qué usar One-Hot Encoding?

Las redes neuronales necesitan que las etiquetas estén en formato one-hot porque:

1. **Evita ordenación implícita**: Si usamos números (0, 1, 2, 3...), el modelo podría interpretar que 3 > 2 > 1, cuando en realidad son categorías sin orden
2. **Compatible con funciones de pérdida**: CrossEntropyLoss y otras funciones trabajan mejor con vectores de probabilidad
3. **Salidas de red**: La última capa de clasificación produce un vector de probabilidades, uno por cada clase

### Ejemplo visual:

Imagina que tenemos 10 clases de ropa en FashionMNIST:

| Etiqueta | Nombre | Representación Numérica | Representación One-Hot |
|----------|--------|------------------------|------------------------|
| 0 | Camiseta | 0 | `[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]` |
| 1 | Pantalón | 1 | `[0, 1, 0, 0, 0, 0, 0, 0, 0, 0]` |
| 2 | Jersey | 2 | `[0, 0, 1, 0, 0, 0, 0, 0, 0, 0]` |
| 3 | Vestido | 3 | `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]` |
| ... | ... | ... | ... |

### Ventajas:
- ✅ No hay relación de orden entre clases
- ✅ Cada clase es igualmente "distante" de las demás
- ✅ Compatible con funciones de activación como Softmax

### Desventajas:
- ❌ Aumenta la dimensionalidad (10 clases → vector de 10 elementos)
- ❌ Usa más memoria
- ❌ Para muchas clases (ej: 1000), los vectores son muy dispersos (muchos ceros)

## ¿Qué son Operaciones In-Place?

**In-Place** (en el lugar) se refiere a operaciones que modifican un tensor directamente en memoria, sin crear una copia nueva.

### Convención en PyTorch:

En PyTorch, las operaciones **in-place** se identifican con un **guión bajo `_` al final del nombre** del método.

| Operación Normal | Operación In-Place | Diferencia |
|------------------|--------------------| -----------|
| `tensor.add(5)` | `tensor.add_(5)` | Normal crea copia, in-place modifica original |
| `tensor.mul(2)` | `tensor.mul_(2)` | Normal crea copia, in-place modifica original |
| `tensor.scatter(...)` | `tensor.scatter_(...)` | Normal crea copia, in-place modifica original |

### Ejemplo comparativo:

```python
# Operación NORMAL (crea un tensor nuevo)
x = torch.tensor([1, 2, 3])
y = x.add(10)  # y es un tensor NUEVO
print(x)  # [1, 2, 3] (sin cambios)
print(y)  # [11, 12, 13] (tensor nuevo)

# Operación IN-PLACE (modifica el tensor existente)
x = torch.tensor([1, 2, 3])
x.add_(10)  # Modifica x directamente
print(x)  # [11, 12, 13] (¡x cambió!)
```

### ¿Por qué usar operaciones in-place?

**Ventajas:**
- ✅ **Ahorro de memoria**: No crea copias adicionales
- ✅ **Mayor velocidad**: Evita asignar nueva memoria
- ✅ **Eficiencia**: Especialmente importante con tensores grandes

**Desventajas:**
- ❌ **Pérdida de datos**: No puedes recuperar el valor original
- ❌ **Problemas con autograd**: Puede interferir con el cálculo de gradientes si se usa incorrectamente
- ❌ **Menos legible**: Modifica variables, puede ser confuso

### Ejemplo con scatter_():

```python
# Usando scatter_ (in-place)
tensor = torch.zeros(5)
tensor.scatter_(0, torch.tensor([2]), 1)
print(tensor)  # [0, 0, 1, 0, 0]
# El tensor original fue modificado

# Si existiera scatter sin _ (no existe, pero como ejemplo)
tensor = torch.zeros(5)
new_tensor = tensor.scatter(0, torch.tensor([2]), 1)  # Hipotético
print(tensor)      # [0, 0, 0, 0, 0] (sin cambios)
print(new_tensor)  # [0, 0, 1, 0, 0] (nuevo tensor)
```

### ⚠️ Advertencia importante:

**NO uses operaciones in-place en tensores que requieren gradientes durante el entrenamiento**, ya que PyTorch necesita los valores originales para calcular los gradientes en la retropropagación.

```python
# ❌ MAL: Puede causar errores en backpropagation
x = torch.tensor([1., 2., 3.], requires_grad=True)
x.add_(5)  # Esto puede causar problemas con autograd

# ✅ BIEN: Usa operación normal
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x.add(5)  # Esto es seguro para autograd
```

In [ ]:
# Ejemplo práctico de One-Hot Encoding
print("="*60)
print("EJEMPLO: ONE-HOT ENCODING")
print("="*60)

# Crear vectores one-hot para 5 clases
num_clases = 5
print(f"\nTenemos {num_clases} clases (0, 1, 2, 3, 4)\n")

for clase in range(num_clases):
    one_hot = torch.zeros(num_clases, dtype=torch.float)
    one_hot[clase] = 1
    print(f"Clase {clase}: {one_hot.numpy()}")

print("\n" + "="*60)
print("EJEMPLO: OPERACIONES IN-PLACE vs NORMALES")
print("="*60)

# Operación NORMAL
print("\n1. Operación NORMAL (sin _):")
x = torch.tensor([1.0, 2.0, 3.0])
print(f"   x original: {x}")
y = x.add(10)  # Crea un tensor nuevo
print(f"   Después de y = x.add(10):")
print(f"   x: {x} (sin cambios)")
print(f"   y: {y} (tensor nuevo)")

# Operación IN-PLACE
print("\n2. Operación IN-PLACE (con _):")
x = torch.tensor([1.0, 2.0, 3.0])
print(f"   x original: {x}")
x.add_(10)  # Modifica x directamente
print(f"   Después de x.add_(10):")
print(f"   x: {x} (¡modificado!)")

print("\n" + "="*60)

## Transformaciones Lambda

Las transformaciones **Lambda** aplican cualquier función lambda definida por el usuario. Aquí definimos una función para convertir el entero en un tensor codificado en **one-hot**.

### ¿Cómo funciona?

La transformación:
1. Crea un tensor de ceros de tamaño 10 (el número de etiquetas en nuestro dataset)
2. Llama a `scatter_()` que asigna un `value=1` en el índice dado por la etiqueta `y`

### Sintaxis de la transformación:

target_transform = Lambda(lambda y: torch.zeros(
    10, dtype=torch.float).scatter_(dim=0, index=torch.tensor(y), value=1))

## Explicación del método scatter_()

`scatter_()` es un método de PyTorch que modifica un tensor **in-place** (en el mismo lugar, sin crear una copia).

### Sintaxis:
```python
tensor.scatter_(dim, index, value)
```

### Parámetros:

- **`dim`** (int): La dimensión a lo largo de la cual indexar
  - `dim=0`: Opera sobre filas (primera dimensión)
  - `dim=1`: Opera sobre columnas (segunda dimensión)

- **`index`** (Tensor): Tensor que contiene los índices de los elementos a modificar
  - Debe ser del mismo tipo que el tensor (LongTensor)
  - Especifica **dónde** colocar los valores

- **`value`** (escalar o Tensor): El valor a insertar en las posiciones especificadas
  - Puede ser un número único que se copiará en todas las posiciones
  - O un tensor con los valores específicos para cada posición

### Ejemplo práctico:

```python
# Crear tensor de ceros
tensor = torch.zeros(10)  # [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

# Colocar 1 en la posición 3
tensor.scatter_(0, torch.tensor([3]), 1)
# Resultado: [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
```

### One-Hot Encoding:

Para convertir una etiqueta numérica en un vector one-hot:
- Si `y = 0` → `[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]`
- Si `y = 3` → `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`
- Si `y = 9` → `[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]`

El método `scatter_()` es perfecto para esto porque:
1. Parte de un vector de ceros
2. Coloca un `1` exactamente en el índice de la clase
3. Deja todos los demás valores en `0`

## Demostración Visual de One-Hot Encoding

In [ ]:
# Demostración: convertir etiquetas numéricas a vectores one-hot
print("Etiqueta → Vector One-Hot\n")
print("-" * 50)

for i in range(10):
    one_hot = torch.zeros(10, dtype=torch.float).scatter_(0, torch.tensor(i), value=1)
    print(f"Clase {i}: {one_hot.numpy()}")

# Ejemplo con datos reales del dataset
print("\n" + "="*50)
print("Ejemplos del dataset FashionMNIST:")
print("="*50)

for i in range(3):
    img, label = ds[i]
    print(f"\nMuestra {i+1}:")
    print(f"  Etiqueta one-hot: {label.numpy()}")
    print(f"  Índice de la clase: {label.argmax().item()}")